In [1]:
### import libraries
import os
import pathlib
import requests
import zipfile
import geopandas as gpd
import holoviews as hv
import hvplot.pandas
from glob import glob
import time
import pandas as pd

### gbif packages
import pygbif.occurrences as occ
import pygbif.species as species
from getpass import getpass

In [94]:
### set up file paths
data_dir = os.path.join(
    pathlib.Path.home(),
    'earth-analytics',
    'data',
    'grasslands'
)

os.makedirs(data_dir, exist_ok=True)

In [95]:
### set a directory for the National Grassland boundary data
boundary_dir = os.path.join(data_dir, 'NG_boundary')

In [96]:
### set the data download URL
boundary_url = "https://data.fs.usda.gov/geodata/edw/edw_resources/shp/BdyDesg_LSRS_NationalGrassland.zip"

# path where zip will be saved
boundary_path = os.path.join(data_dir, "national_grassland.zip")

# download file
response = requests.get(boundary_url, stream=True)
response.raise_for_status()

with open(boundary_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f"Downloaded to: {boundary_path}")

Downloaded to: C:\Users\warno\earth-analytics\data\grasslands\national_grassland.zip


In [97]:
# make sure boundary_dir exists
os.makedirs(boundary_dir, exist_ok=True)

### unzip the file
with zipfile.ZipFile(boundary_path, 'r') as zip_ref:
    zip_ref.extractall(boundary_dir)

print(f"Extracted to: {boundary_dir}")

Extracted to: C:\Users\warno\earth-analytics\data\grasslands\NG_boundary


In [98]:
### automatically check for .shp file name
shp_files = [f for f in os.listdir(boundary_dir) if f.endswith(".shp")]

shp_path = os.path.join(boundary_dir, shp_files[0])
boundary_gdf = gpd.read_file(shp_path)

In [99]:
# select only Pawnee National Grassland
pawnee = boundary_gdf[boundary_gdf["GRASSLANDN"] == "Pawnee National Grassland"].copy()

# check the result
pawnee

,NATIONALGR,GRASSLANDN,GIS_ACRES,SHAPE_AREA,SHAPE_LEN,geometry
0,295523010328,Pawnee National Grassland,208424.885,0.089972,15.341594,"MULTIPOLYGON (((-104.58106 40.82664, -104.5810..."


In [100]:
# path for new shapefile
pawnee_shp = os.path.join(boundary_dir, "pawnee_national_grassland.shp")

# write to shapefile
pawnee.to_file(pawnee_shp)

print(pawnee_shp)

INFO:Created 1 records


C:\Users\warno\earth-analytics\data\grasslands\NG_boundary\pawnee_national_grassland.shp


In [101]:
pawnee.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Pawnee National Grassland Boundaries',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

### GBIF Login

In [3]:
### reset credentials
reset_credentials = False

### make dictionary for GBIF username and pass
credentials = dict(
    GBIF_USER=(input, 'GBIF username:'),
    GBIF_PWD=(getpass, 'GBIF password'),
    GBIF_EMAIL=(input, 'GBIF email'),
)

### loop through credentials and enter them
for env_variable, (prompt_func, prompt_text) in credentials.items():

    if reset_credentials and (env_variable in os.environ):
        os.environ.pop(env_variable)

    if not env_variable in os.environ:
        os.environ[env_variable] = prompt_func(prompt_text)

### Download GBIF data for Prairie Dogs and Pronghorn

In [ ]:
### function to download GBIF data for Prairie Dogs and Pronghorn
def download_gbif_species(species_name, folder_name, data_dir):
    """
    Download one GBIF species file, extract it, and read it into a DataFrame.
    """
    ### search species in GBIF backbone
    species_info = species.name_backbone(name=species_name)

    ### get the usage key
    species_key = species_info["usageKey"]
    print(f"{species_name}: {species_key}")

    ### set species directory
    gbif_dir = os.path.join(data_dir, folder_name)
    os.makedirs(gbif_dir, exist_ok=True)

    ### make a file path pattern for data
    gbif_pattern = os.path.join(gbif_dir, "*.csv")

    ### download it once
    if not glob(gbif_pattern):

        ### submit query
        gbif_query = occ.download([
            f"speciesKey = {species_key}",
            "hasCoordinate = True",
        ])

        download_key = gbif_query[0]

        ### wait for the download to build
        wait = occ.download_meta(download_key)["status"]
        while wait != "SUCCEEDED":
            if wait in {"CANCELLED", "KILLED", "FAILED"}:
                raise RuntimeError(f"GBIF download failed for {species_name}: {wait}")
            time.sleep(5)
            wait = occ.download_meta(download_key)["status"]

        ### download data
        download_info = occ.download_get(
            download_key,
            path=gbif_dir
        )

        ### unzip the file
        with zipfile.ZipFile(download_info["path"]) as download_zip:
            download_zip.extractall(path=gbif_dir)

    ### find csv file path
    gbif_path = glob(gbif_pattern)[0]

    ### read csv
    gbif_df = pd.read_csv(
        gbif_path,
        delimiter="\t",
        low_memory=False
    )

    return gbif_df, gbif_path, species_key

In [91]:
species_to_download = [
    {
        "species_name": "Antilocapra americana",
        "folder_name": "gbif_pronghorn"
    },
    {
        "species_name": "Cynomys ludovicianus",
        "folder_name": "gbif_prairie_dog"
    }
]

gbif_data = {}

for item in species_to_download:
    gbif_df, gbif_path, species_key = download_gbif_species(
        species_name=item["species_name"],
        folder_name=item["folder_name"],
        data_dir=data_dir
    )

    gbif_data[item["species_name"]] = {
        "df": gbif_df,
        "path": gbif_path,
        "species_key": species_key
    }

    print(f"\nLoaded {item['species_name']}")
    print(f"Path: {gbif_path}")
    print(gbif_df.head())

Antilocapra americana: 2440902


INFO:Your download key is 0080121-260226173443078
INFO:Download file size: 1198147 bytes
INFO:On disk at C:\Users\warno\earth-analytics\data\grasslands\gbif_pronghorn/0080121-260226173443078.zip



Loaded Antilocapra americana
Path: C:\Users\warno\earth-analytics\data\grasslands\gbif_pronghorn\0080121-260226173443078.csv
      gbifID                            datasetKey  \
0  930740086  0096dfc0-9925-47ef-9700-9b77814295f1   
1  923922129  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
2  923922121  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
3  923920930  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
4  923918563  50c9509d-22c7-4a22-a47d-8c48425ef4a7   

                                        occurrenceID   kingdom    phylum  \
0  http://bioimages.vanderbilt.edu/ind-baskauf/14...  Animalia  Chordata   
1     http://www.inaturalist.org/observations/752321  Animalia  Chordata   
2     http://www.inaturalist.org/observations/752191  Animalia  Chordata   
3     http://www.inaturalist.org/observations/749240  Animalia  Chordata   
4     http://www.inaturalist.org/observations/743215  Animalia  Chordata   

      class         order          family        genus                species  \
0  Mammalia

INFO:Your download key is 0078011-260226173443078
INFO:Download file size: 821468 bytes
INFO:On disk at C:\Users\warno\earth-analytics\data\grasslands\gbif_prairie_dog/0078011-260226173443078.zip



Loaded Cynomys ludovicianus
Path: C:\Users\warno\earth-analytics\data\grasslands\gbif_prairie_dog\0078011-260226173443078.csv
      gbifID                            datasetKey  \
0   92891164  b929f23d-290f-4e85-8f17-764c55b3b284   
1  923925236  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
2  919448871  22a66350-7947-4a49-84a3-39c7c1b0881f   
3  919447105  22a66350-7947-4a49-84a3-39c7c1b0881f   
4  919439868  22a66350-7947-4a49-84a3-39c7c1b0881f   

                                     occurrenceID   kingdom    phylum  \
0            4d8ec95f-e671-46a5-a113-6f6adb97f836  Animalia  Chordata   
1  http://www.inaturalist.org/observations/761318  Animalia  Chordata   
2                      urn:catalog:MSU:MR:MR.9308  Animalia  Chordata   
3                      urn:catalog:MSU:MR:MR.9307  Animalia  Chordata   
4                      urn:catalog:MSU:MR:MR.9306  Animalia  Chordata   

      class     order     family    genus               species  ...  \
0  Mammalia  Rodentia  Sciuridae  Cyn

In [92]:
gbif_pronghorn_df = gbif_data["Antilocapra americana"]["df"]
gbif_prairie_dog_df = gbif_data["Cynomys ludovicianus"]["df"]